# Uji kode pendamping buku *Data Mining dengan Python*

Notebook ini menguji seluruh kode di repositori
[bsnegara/data-mining-dengan-python](https://github.com/bsnegara/data-mining-dengan-python)
di lingkungan Kaggle yang bersih:

1. menyalin (clone) repositori dan memasang pustaka dari `kode/requirements.txt`;
2. menjalankan setiap skrip `kode/babNN/*.py` dan membandingkan keluarannya dengan berkas `.out` (keluaran yang dicetak di buku);
3. menjalankan setiap notebook di `kode/notebook/`;
4. membangun ulang seluruh data dari skrip pembangkit dan memeriksa apakah hasilnya identik dengan CSV di repositori.

**Cara pakai:** pastikan *Internet* menyala (panel kanan, *Session options* → *Internet on*), lalu klik **Run All**.
Seluruh uji memakan waktu kira-kira 20–40 menit. Di akhir, sel terakhir mencetak **RINGKASAN** yang cukup disalin
dan ditempelkan kembali ke Claude. Laporan lengkap tersimpan di `/kaggle/working/hasil_uji/`.

Arti status:
- **SAMA**: keluaran persis sama dengan buku.
- **MIRIP**: hanya beda kecil pada angka (dalam toleransi) atau pada baris waktu/durasi.
- **BEDA**: teks atau angka berbeda melebihi toleransi; contoh baris yang berbeda dicantumkan.
- **GALAT**: skrip berhenti dengan pesan galat (error); baris terakhir pesan galat dicantumkan.
- **WAKTU HABIS**: skrip melewati batas waktu.

In [ ]:
# ===== PENGATURAN (boleh diubah) =====
REPO = "https://github.com/bsnegara/data-mining-dengan-python"
CABANG = "main"            # cabang (branch) repositori yang diuji
BAB = None                 # None = semua bab; atau misalnya [8, 12, 27]
JALANKAN_SKRIP = True      # uji skrip babNN/*.py terhadap .out
JALANKAN_NOTEBOOK = True   # jalankan notebook kode/notebook/*.ipynb
CEK_DATA = True            # bangun ulang data dan bandingkan dengan CSV di repo
TOLERANSI_RELATIF = 0.02   # beda angka <= 2% dianggap MIRIP
BATAS_WAKTU = 1800         # detik per skrip/notebook
BUAT_FOLDER_GAMBAR = False # True = buat folder gambar/babNN dulu (banyak skrip menulis data gambar ke sana)
# ======================================

import os, sys, re, json, time, shutil, hashlib, platform, subprocess
from pathlib import Path

KERJA = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
AKAR = KERJA / "data-mining-dengan-python"
HASIL = KERJA / "hasil_uji"
HASIL.mkdir(exist_ok=True)
MULAI = time.time()
print("Folder kerja :", KERJA)
print("Python       :", sys.version.split()[0], "|", platform.platform())

## 1. Salin repositori dan pasang pustaka

Pemasangan dicoba bertahap: (A) versi persis `requirements.txt`; bila gagal, (B) versi persis dengan
perbaikan bentrok yang diketahui (pandas 2.3.3, karena mlxtend 0.25 menuntut pandas ≥ 2.3.3);
bila masih gagal, (C) hanya memasang pustaka yang belum ada di Kaggle (`requirements-colab.txt`)
dan memakai versi bawaan Kaggle. Mode yang terpakai dicatat di ringkasan.

In [ ]:
if AKAR.exists():
    shutil.rmtree(AKAR)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", CABANG, REPO, str(AKAR)],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
assert (AKAR / "kode").exists(), "Gagal menyalin repositori. Apakah Internet sudah menyala?"
KOMIT = subprocess.run(["git", "-C", str(AKAR), "log", "-1", "--format=%h %s"],
                       capture_output=True, text=True).stdout.strip()
print("Komit diuji  :", KOMIT)
if BUAT_FOLDER_GAMBAR:
    for nn in range(1, 29):
        (AKAR / "gambar" / f"bab{nn:02d}").mkdir(parents=True, exist_ok=True)
    print("Folder gambar/bab01..bab28 dibuat.")

CPU = ["--extra-index-url", "https://download.pytorch.org/whl/cpu"]  # torch versi CPU, lebih kecil


def pasang(berkas, tambahan=()):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(berkas), *tambahan]
    r = subprocess.run(cmd, capture_output=True, text=True)
    (HASIL / "pip.log").open("a").write(f"\n$ {' '.join(cmd)}\n{r.stdout}\n{r.stderr}\n")
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]


req = AKAR / "kode" / "requirements.txt"
ok, pesan = pasang(req, CPU)
MODE_PASANG = "A: versi persis requirements.txt"
if not ok:
    print("Mode A gagal:", pesan)
    req_b = HASIL / "requirements_b.txt"
    req_b.write_text(re.sub(r"(?m)^pandas==2\.2\.\d+", "pandas==2.3.3", req.read_text()))
    ok, pesan = pasang(req_b, CPU)
    MODE_PASANG = "B: versi persis, pandas==2.3.3"
if not ok:
    print("Mode B gagal:", pesan)
    ok, pesan = pasang(AKAR / "kode" / "requirements-colab.txt")
    MODE_PASANG = "C: versi bawaan Kaggle + requirements-colab.txt"
if not ok:
    print("Mode C gagal:", pesan)
    MODE_PASANG += " (GAGAL, lihat hasil_uji/pip.log)"
print("Mode pasang  :", MODE_PASANG)

# Versi pustaka yang benar-benar terpasang dibanding requirements.txt
kode_versi = (
    "import importlib.metadata as m, re, sys\n"
    "for b in open(sys.argv[1]):\n"
    "    b = b.split('#')[0].strip()\n"
    "    if '==' not in b: continue\n"
    "    n, v = b.split('==')\n"
    "    try: a = m.version(n)\n"
    "    except Exception: a = 'TIDAK ADA'\n"
    "    print(f'{n}|{v}|{a}')\n")
r = subprocess.run([sys.executable, "-c", kode_versi, str(req)], capture_output=True, text=True)
VERSI = [b.split("|") for b in r.stdout.split()]
BEDA_VERSI = [f"{n} {a} (buku {v})" for n, v, a in VERSI if a != v and not a.startswith(v + "+")]
for n, v, a in VERSI:
    print(f"  {n:15s} buku {v:10s} terpasang {a}")

## 2. Fungsi pembanding keluaran

In [ ]:
ANGKA = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?")
# Baris yang memuat kata-kata ini dianggap baris waktu: angkanya tidak dibandingkan.
WAKTU = re.compile(r"(detik|\bdtk\b|\bms\b|\d\s*s\b|\bsec|waktu|durasi|elapsed|it/s|s/it|"
                   r"\d{2}:\d{2}:\d{2})", re.I)


def rapikan(teks):
    baris = [b.rstrip() for b in teks.replace("\r", "").split("\n")]
    while baris and not baris[-1]:
        baris.pop()
    return baris


def bandingkan(harap, dapat, tol=TOLERANSI_RELATIF):
    """Kembalikan (status, catatan). status: SAMA, MIRIP, BEDA."""
    a, b = rapikan(harap), rapikan(dapat)
    if a == b:
        return "SAMA", ""
    if sorted(a) == sorted(b):
        return "MIRIP", "baris sama, urutan berbeda"
    if len(a) != len(b):
        # cari baris pertama yang berbeda
        for i, (x, y) in enumerate(zip(a, b)):
            if ANGKA.sub("#", x) != ANGKA.sub("#", y):
                break
        else:
            i = min(len(a), len(b))
        x = a[i] if i < len(a) else "<tidak ada>"
        y = b[i] if i < len(b) else "<tidak ada>"
        return "BEDA", f"jumlah baris {len(a)} vs {len(b)}; baris {i+1}: buku «{x.strip()}» | kini «{y.strip()}»"
    maks, contoh, teks_beda = 0.0, None, None
    for i, (x, y) in enumerate(zip(a, b)):
        if x == y:
            continue
        if ANGKA.sub("#", x).split() != ANGKA.sub("#", y).split():
            teks_beda = teks_beda or (i, x, y)
            continue
        if WAKTU.search(x):
            continue
        for sp, sq in zip(ANGKA.findall(x), ANGKA.findall(y)):
            p, q = float(sp), float(sq)
            if p == q:
                continue
            # beda 1 satuan digit terakhir yang dicetak (pembulatan) selalu diterima
            des = len(sp.split(".")[1].split("e")[0].split("E")[0]) if "." in sp else 0
            if "e" not in sp.lower() and abs(p - q) <= 1.01 * 10 ** -des:
                maks = max(maks, 1e-9)
                continue
            rel = abs(p - q) / max(abs(p), abs(q), 1e-12)
            if rel > maks:
                maks, contoh = rel, (i, x, y)
    if teks_beda:
        i, x, y = teks_beda
        return "BEDA", f"teks baris {i+1}: buku «{x.strip()}» | kini «{y.strip()}»"
    if maks <= tol:
        return "MIRIP", (f"beda angka maks {maks:.2%}" if maks else "hanya baris waktu")
    i, x, y = contoh
    return "BEDA", f"angka beda {maks:.1%} di baris {i+1}: buku «{x.strip()}» | kini «{y.strip()}»"


def ekor_galat(teks, n=1):
    teks = re.sub(r"\x1b\[[0-9;]*m", "", teks)      # buang kode warna
    baris = [b for b in rapikan(teks) if b.strip()]
    for b in reversed(baris):
        if re.match(r"^\w*(Error|Exception)\b", b) or "Error:" in b:
            return b.strip()[:300]
    return " / ".join(x.strip() for x in baris[-n:])[:300]


lingkungan = dict(os.environ, PYTHONWARNINGS="ignore", MPLBACKEND="Agg")
bab_dipilih = lambda nn: BAB is None or int(nn) in BAB
print("Siap.")

## 3. Jalankan setiap skrip bab dan bandingkan dengan `.out`

Skrip dijalankan dari akar repositori dengan urutan abjad per bab (sama dengan cara `.out`
dibuat). Skrip berawalan `_` (pembuat bahan gambar buku) dan modul bantu tanpa `.out` dilewati.

In [ ]:
HASIL_SKRIP = []
if JALANKAN_SKRIP:
    for folder in sorted((AKAR / "kode").glob("bab[0-9][0-9]")):
        nn = folder.name[3:]
        if not bab_dipilih(nn):
            continue
        for skrip in sorted(folder.glob("[!_]*.py")):
            out = skrip.with_suffix(".out")
            if not out.exists():
                continue
            rel = skrip.relative_to(AKAR / "kode")
            t0 = time.time()
            try:
                r = subprocess.run([sys.executable, str(skrip.relative_to(AKAR))], cwd=AKAR,
                                   env=lingkungan, capture_output=True, text=True,
                                   timeout=BATAS_WAKTU)
                keluaran = r.stdout + r.stderr
                if r.returncode != 0:
                    status, catatan = "GALAT", ekor_galat(keluaran)
                else:
                    status, catatan = bandingkan(out.read_text(), keluaran)
            except subprocess.TimeoutExpired as e:
                keluaran = (e.stdout or b"").decode(errors="replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
                status, catatan = "WAKTU HABIS", f"> {BATAS_WAKTU} detik"
            dt = time.time() - t0
            simpan = HASIL / "keluaran" / rel.with_suffix(".out")
            simpan.parent.mkdir(parents=True, exist_ok=True)
            simpan.write_text(keluaran)
            HASIL_SKRIP.append(dict(bab=nn, berkas=str(rel), status=status, catatan=catatan, detik=round(dt, 1)))
            print(f"{status:11s} {str(rel):40s} {dt:6.1f} dtk  {catatan[:120]}")

## 4. Jalankan setiap notebook

Notebook dijalankan dari foldernya sendiri (sel pertama notebook mencari akar repositori).
Yang diuji: semua sel berjalan tanpa galat.

In [ ]:
HASIL_NB = []
if JALANKAN_NOTEBOOK:
    import nbformat
    from nbclient import NotebookClient
    from nbclient.exceptions import CellExecutionError, CellTimeoutError, DeadKernelError
    os.environ.update(PYTHONWARNINGS="ignore", MPLBACKEND="Agg")
    folder_nb = AKAR / "kode" / "notebook"
    (HASIL / "notebook").mkdir(exist_ok=True)
    for berkas in sorted(folder_nb.glob("bab*.ipynb")):
        nn = berkas.name[3:5]
        if not bab_dipilih(nn):
            continue
        nb = nbformat.read(berkas, as_version=4)
        klien = NotebookClient(nb, timeout=BATAS_WAKTU, kernel_name="python3",
                               resources={"metadata": {"path": str(folder_nb)}})
        t0 = time.time()
        try:
            klien.execute()
            status, catatan = "LULUS", ""
        except CellExecutionError as e:
            status, catatan = "GALAT", ekor_galat(str(e))
        except (CellTimeoutError, TimeoutError):
            status, catatan = "WAKTU HABIS", f"satu sel > {BATAS_WAKTU} detik"
        except DeadKernelError as e:
            status, catatan = "GALAT", f"kernel mati: {str(e)[:200]}"
        except Exception as e:
            status, catatan = "GALAT", f"{type(e).__name__}: {str(e)[:200]}"
        dt = time.time() - t0
        nbformat.write(nb, HASIL / "notebook" / berkas.name)
        HASIL_NB.append(dict(bab=nn, berkas=berkas.name, status=status, catatan=catatan, detik=round(dt, 1)))
        print(f"{status:11s} {berkas.name:45s} {dt:6.1f} dtk  {catatan[:120]}")

## 5. Bangun ulang data dan bandingkan dengan CSV di repositori

Rantai pembangun dijalankan di salinan terpisah (urutan sesuai `kode/README.md`), lalu setiap
CSV dibandingkan sidik jari (hash)-nya dengan CSV di repositori.

In [ ]:
HASIL_DATA = []
if CEK_DATA:
    salinan = KERJA / "salinan_data"
    if salinan.exists():
        shutil.rmtree(salinan)
    shutil.copytree(AKAR, salinan, ignore=shutil.ignore_patterns(".git"))
    for csv in (salinan / "kode" / "data").glob("*.csv"):
        csv.unlink()                       # mulai dari nol
    rantai = [
        ["kode/data/bangkitkan_data.py", "kode/data"],
        ["kode/bab03/seragamkan.py"],
        ["kode/bab03/gabung_dimensi.py"],
        ["kode/data/bangkitkan_data_lanjut.py", "kode/data"],
        ["kode/data/bangkitkan_data_rekomendasi.py", "kode/data"],
        ["kode/data/bangkitkan_data_mds.py", "kode/data"],
        ["kode/data/bangkitkan_data_proyek.py", "kode/data"],
        ["kode/bab05/bersihkan.py"],
    ]
    rantai_ok = True
    for langkah in rantai:
        t0 = time.time()
        r = subprocess.run([sys.executable, *langkah], cwd=salinan, env=lingkungan,
                           capture_output=True, text=True, timeout=BATAS_WAKTU)
        print(f"{'OK' if r.returncode == 0 else 'GALAT':6s} {' '.join(langkah):50s} {time.time()-t0:6.1f} dtk")
        if r.returncode != 0:
            rantai_ok = False
            HASIL_DATA.append(dict(berkas=langkah[0], status="GALAT", catatan=ekor_galat(r.stdout + r.stderr)))
            break
    sidik = lambda p: hashlib.md5(p.read_bytes()).hexdigest()
    for asli in sorted((AKAR / "kode" / "data").glob("*.csv")):
        baru = salinan / "kode" / "data" / asli.name
        if not baru.exists():
            status, catatan = "TIDAK TERBENTUK", ""
        elif sidik(asli) == sidik(baru):
            status, catatan = "IDENTIK", ""
        else:
            import pandas as pd
            try:
                x, y = pd.read_csv(asli), pd.read_csv(baru)
                if x.shape != y.shape:
                    status, catatan = "BEDA", f"ukuran {x.shape} vs {y.shape}"
                else:
                    kol = [c for c in x.columns if not x[c].equals(y[c])]
                    status, catatan = "BEDA", f"kolom berbeda: {', '.join(map(str, kol[:6]))}"
            except Exception as e:
                status, catatan = "BEDA", f"{type(e).__name__}"
        HASIL_DATA.append(dict(berkas=asli.name, status=status, catatan=catatan))
        print(f"{status:15s} {asli.name:35s} {catatan}")

## 6. Ringkasan

Salin seluruh teks di bawah garis `=== RINGKASAN ===` dan tempelkan ke Claude.
Laporan lengkap: `/kaggle/working/hasil_uji/laporan.md` (unduh dari panel *Output*).

In [ ]:
from collections import Counter, defaultdict

def hitung(daftar):
    c = Counter(d["status"] for d in daftar)
    return ", ".join(f"{k} {v}" for k, v in sorted(c.items())) or "-"

L = []
L.append("=== RINGKASAN UJI KODE BUKU (Kaggle) ===")
L.append(f"Repo {REPO} @ {CABANG} | komit {KOMIT}")
L.append(f"Python {sys.version.split()[0]} | pasang: {MODE_PASANG} | total {(time.time()-MULAI)/60:.0f} menit"
         f" | folder gambar dibuat: {'ya' if BUAT_FOLDER_GAMBAR else 'tidak'}")
L.append("Versi beda dari buku: " + ("; ".join(BEDA_VERSI) if BEDA_VERSI else "tidak ada"))
if JALANKAN_SKRIP:
    L.append(f"\nSKRIP ({len(HASIL_SKRIP)}): {hitung(HASIL_SKRIP)}")
    per_bab = defaultdict(list)
    for d in HASIL_SKRIP:
        per_bab[d["bab"]].append(d)
    for nn, ds in per_bab.items():
        c = Counter(d["status"] for d in ds)
        L.append(f" Bab {nn}: " + ", ".join(f"{k} {v}" for k, v in sorted(c.items())))
    masalah = [d for d in HASIL_SKRIP if d["status"] != "SAMA"]
    if masalah:
        L.append(" Rincian selain SAMA:")
        for d in masalah:
            L.append(f"  - [{d['status']}] {d['berkas']}: {d['catatan'][:220]}")
if JALANKAN_NOTEBOOK:
    L.append(f"\nNOTEBOOK ({len(HASIL_NB)}): {hitung(HASIL_NB)}")
    for d in HASIL_NB:
        if d["status"] != "LULUS":
            L.append(f"  - [{d['status']}] {d['berkas']}: {d['catatan'][:220]}")
if CEK_DATA:
    L.append(f"\nDATA ({len(HASIL_DATA)}): {hitung(HASIL_DATA)}")
    for d in HASIL_DATA:
        if d["status"] != "IDENTIK":
            L.append(f"  - [{d['status']}] {d['berkas']}: {d['catatan'][:220]}")
L.append("=== AKHIR RINGKASAN ===")
ringkasan = "\n".join(L)
print(ringkasan)

# Laporan lengkap
md = [ringkasan.replace("\n", "  \n"), "\n## Semua skrip\n", "| Status | Berkas | Detik | Catatan |", "|---|---|---:|---|"]
md += [f"| {d['status']} | {d['berkas']} | {d['detik']} | {d['catatan'].replace('|', '/')} |" for d in HASIL_SKRIP]
md += ["\n## Notebook\n", "| Status | Berkas | Detik | Catatan |", "|---|---|---:|---|"]
md += [f"| {d['status']} | {d['berkas']} | {d['detik']} | {d['catatan'].replace('|', '/')} |" for d in HASIL_NB]
md += ["\n## Versi pustaka\n", "| Pustaka | Buku | Terpasang |", "|---|---|---|"]
md += [f"| {n} | {v} | {a} |" for n, v, a in VERSI]
(HASIL / "ringkasan.txt").write_text(ringkasan + "\n")
(HASIL / "laporan.md").write_text("\n".join(md) + "\n")
json.dump(dict(skrip=HASIL_SKRIP, notebook=HASIL_NB, data=HASIL_DATA, versi=VERSI,
               mode=MODE_PASANG, komit=KOMIT), open(HASIL / "hasil.json", "w"), indent=1)
# bersihkan salinan besar agar panel Output tidak penuh
for p in (KERJA / "salinan_data", AKAR):
    shutil.rmtree(p, ignore_errors=True)
print("\nLaporan lengkap tersimpan di", HASIL)